In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, precision_recall_curve
)
from xgboost import XGBClassifier

from src.dataset import OpenBanditDataset
from obp.ope import OffPolicyEvaluation, DirectMethod

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

POLICY_RANDOM       = "random"
POLICY_BTS          = "bts"
CAMPAIGN            = "all"
N_ACTIONS           = 80
LEN_LIST            = 3
N_CONTEXT_FEATURES  = 20   # first 20 features common to both policies
RANDOM_STATE        = 42
FIGURES_DIR         = Path("../figures/week2")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
dataset_random  = OpenBanditDataset(behavior_policy=POLICY_RANDOM, campaign=CAMPAIGN)
feedback_random = dataset_random.obtain_batch_bandit_feedback()

dataset_bts     = OpenBanditDataset(behavior_policy=POLICY_BTS, campaign=CAMPAIGN)
feedback_bts    = dataset_bts.obtain_batch_bandit_feedback()

for name, fb in [(POLICY_RANDOM, feedback_random), (POLICY_BTS, feedback_bts)]:
    print(f"[{name}]  context shape: {fb['context'].shape}  "
          f"n_rounds: {fb['n_rounds']}  CTR: {fb['reward'].mean():.4f}")


In [ ]:

context_random = feedback_random["context"][:, :N_CONTEXT_FEATURES]
action_random  = feedback_random["action"]
reward_random  = feedback_random["reward"]

action_onehot  = np.eye(N_ACTIONS)[action_random]
X_train_full   = np.hstack([context_random, action_onehot])
y_train_full   = reward_random.astype(np.float32)

print(f"X_train.shape : {X_train_full.shape}")
n_pos = int(y_train_full.sum())
n_neg = len(y_train_full) - n_pos
print(f"y_train: 0 -> {n_neg} ({n_neg/len(y_train_full)*100:.2f}%),  "
      f"1 -> {n_pos} ({n_pos/len(y_train_full)*100:.2f}%)")
print(f"CTR (reward=1): {y_train_full.mean():.4f}")


In [ ]:
n_positive = int(y_train_full.sum())
n_negative = len(y_train_full) - n_positive

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_full
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=n_negative / n_positive,
    eval_metric="logloss",
    early_stopping_rounds=20,
    random_state=RANDOM_STATE,
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

y_pred_proba = model.predict_proba(X_val)[:, 1]

val_logloss = log_loss(y_val, y_pred_proba)
val_auc_roc = roc_auc_score(y_val, y_pred_proba)
val_auc_pr  = average_precision_score(y_val, y_pred_proba)

print(f"Val LogLoss : {val_logloss:.6f}")
print(f"Val AUC-ROC : {val_auc_roc:.4f}")
print(f"Val AUC-PR  : {val_auc_pr:.4f}")


In [ ]:
baseline_ctr = float(y_train_full.mean())

precision, recall, _ = precision_recall_curve(y_val, y_pred_proba)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, color="#1f77b4", lw=2,
        label=f"PR Curve  (AUC-PR = {val_auc_pr:.4f})")
ax.axhline(baseline_ctr, color="crimson", ls="--", lw=1.8,
           label=f"Baseline CTR = {baseline_ctr:.4f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — Q-hat trained on random policy data")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(y_pred_proba[y_val == 0], bins=60, alpha=0.6,
        color="#1f77b4", density=True, label="reward = 0")
ax.hist(y_pred_proba[y_val == 1], bins=60, alpha=0.7,
        color="#ff7f0e", density=True, label="reward = 1")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Density")
ax.set_title("Rozkład predicted proba — czy model rozdziela klasy?")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "score_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
context_bts  = feedback_bts["context"][:, :N_CONTEXT_FEATURES]
n_rounds_bts = int(feedback_bts["n_rounds"])

expected_reward = np.zeros((n_rounds_bts, N_ACTIONS), dtype=np.float32)

for a in range(N_ACTIONS):
    action_onehot_a = np.tile(np.eye(N_ACTIONS, dtype=np.float32)[a], (n_rounds_bts, 1))
    X_a = np.hstack([context_bts, action_onehot_a])
    expected_reward[:, a] = model.predict_proba(X_a)[:, 1]

print(f"expected_reward.shape : {expected_reward.shape}")
print(f"expected_reward[:3, :5] :")
print(expected_reward[:3, :5])


In [ ]:
expected_reward_3d = np.tile(
    expected_reward[:, :, np.newaxis], (1, 1, LEN_LIST)
).astype(np.float64)

positions_bts    = feedback_bts["position"].astype(int)
actions_bts      = feedback_bts["action"].astype(int)
idx              = np.arange(n_rounds_bts)

action_dist_bts  = np.ones((n_rounds_bts, N_ACTIONS, LEN_LIST), dtype=np.float64) / N_ACTIONS
action_dist_bts[idx, :, positions_bts] = 0.0
action_dist_bts[idx, actions_bts, positions_bts] = 1.0
action_dist_random_bl = np.full(
    (n_rounds_bts, N_ACTIONS, LEN_LIST), 1.0 / N_ACTIONS
)

dm = DirectMethod()

v_dm_bts    = dm.estimate_policy_value(
    action_dist=action_dist_bts,
    estimated_rewards_by_reg_model=expected_reward_3d,
    position=positions_bts,
)
v_dm_random = dm.estimate_policy_value(
    action_dist=action_dist_random_bl,
    estimated_rewards_by_reg_model=expected_reward_3d,
    position=positions_bts,
)

ope_bts = OffPolicyEvaluation(
    bandit_feedback=feedback_bts,
    ope_estimators=[DirectMethod()]
)
ci_bts    = ope_bts.estimate_intervals(
    action_dist=action_dist_bts,
    estimated_rewards_by_reg_model=expected_reward_3d,
    n_bootstrap_samples=1000,
    random_state=RANDOM_STATE,
)
ci_random = ope_bts.estimate_intervals(
    action_dist=action_dist_random_bl,
    estimated_rewards_by_reg_model=expected_reward_3d,
    n_bootstrap_samples=1000,
    random_state=RANDOM_STATE,
)

ci_key_lo     = "95.0% CI (lower)"
ci_key_hi     = "95.0% CI (upper)"
estimator_key = "dm"

results = {
    "random (baseline)": {
        "V_DM":  v_dm_random,
        "CI_lo": ci_random[estimator_key][ci_key_lo],
        "CI_hi": ci_random[estimator_key][ci_key_hi],
    },
    "bts": {
        "V_DM":  v_dm_bts,
        "CI_lo": ci_bts[estimator_key][ci_key_lo],
        "CI_hi": ci_bts[estimator_key][ci_key_hi],
    },
}

print("=" * 62)
print(f"{'Polityka':<22} {'V_DM':>10}  {'95% CI':>25}")
print("=" * 62)
for pol, vals in results.items():
    ci_str = f"[{vals['CI_lo']:.6f}, {vals['CI_hi']:.6f}]"
    print(f"{pol:<22} {vals['V_DM']:>10.6f}  {ci_str:>25}")
print("=" * 62)


In [ ]:
NAIVE_CTR_RANDOM = 0.0038
NAIVE_CTR_BTS    = 0.0042

labels    = [POLICY_RANDOM, POLICY_BTS]
naive_ctr = [NAIVE_CTR_RANDOM, NAIVE_CTR_BTS]
v_dm      = [results["random (baseline)"]["V_DM"], results["bts"]["V_DM"]]

x      = np.arange(len(labels))
width  = 0.35

fig, ax = plt.subplots(figsize=(9, 6))

bars_naive = ax.bar(x - width / 2, naive_ctr, width, label="Naive CTR",
                    color=["#5B9BD5", "#ED7D31"], alpha=0.85)
bars_dm    = ax.bar(x + width / 2, v_dm, width, label="V_DM (Direct Method)",
                    color=["#2E75B6", "#C55A11"], alpha=0.85)

for bar in bars_naive:
    h = bar.get_height()
    ax.annotate(f"{h:.4f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)

for i, bar in enumerate(bars_dm):
    h = bar.get_height()
    ci_lo = results[list(results.keys())[i]]["CI_lo"]
    ci_hi = results[list(results.keys())[i]]["CI_hi"]
    ax.annotate(f"{h:.4f}\n[{ci_lo:.4f},{ci_hi:.4f}]",
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel("Estimated policy value")
ax.set_title(
    "Naive CTR vs Direct Method — czy DM koryguje bias?",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=11)
ax.set_ylim(0, max(max(naive_ctr), max(v_dm)) * 1.4)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "dm_vs_naive.png", dpi=150, bbox_inches="tight")
plt.show()
